# Working with JuPedSim Web-UI Scenarios in Python

Load scenario JSON files exported from the web UI and run them programmatically.

**Workflow:**
1. Design your scenario in the web UI (geometry, exits, distributions, etc.)
2. Export the scenario JSON
3. Load it here, inspect/modify, run, and analyze

**Install dependencies:**
```bash
pip install -r requirements.txt
```

In [ ]:
from jupedsim_scenario import load_scenario, run_scenario

## 1. Load and inspect a scenario

In [ ]:
# Point this at any JSON exported from the web UI
scenario = load_scenario("scenarios/corridor_simple.json")
print(scenario.summary())

## 2. Modify parameters before running

In [ ]:
scenario.set_agent_count("jps-distributions_0", 20)
scenario.set_max_time(60)
print(scenario.summary())

## 3. Run the simulation

In [ ]:
result = run_scenario(scenario)

print(f"Success:          {result.success}")
print(f"Evacuation time:  {result.evacuation_time:.2f}s")
print(f"Total agents:     {result.total_agents}")
print(f"Evacuated:        {result.agents_evacuated}")
print(f"Remaining:        {result.agents_remaining}")

## 4. Analyze trajectory data

In [ ]:
df = result.trajectory_dataframe()
print(f"Trajectory: {len(df)} rows, {df['id'].nunique()} agents, {df['frame'].nunique()} frames")
df.head(10)

## 5. Plot trajectories

In [ ]:
import pedpy

traj = pedpy.TrajectoryData(df, frame_rate=result.frame_rate)
walkable_area = pedpy.WalkableArea(scenario.walkable_polygon)

pedpy.plot_trajectories(
    walkable_area=walkable_area,
    traj=traj,
).set_title("Agent trajectories")

## 6. Parameter sweep — compare seeds

In [ ]:
seeds = [1, 2, 3, 4, 5]
evac_times = []

for s in seeds:
    r = run_scenario(scenario, seed=s)
    evac_times.append(r.evacuation_time)
    r.cleanup()
    print(f"  Seed {s}: {r.evacuation_time:.2f}s")

print(f"\nMean: {sum(evac_times)/len(evac_times):.2f}s, "
      f"Min: {min(evac_times):.2f}s, Max: {max(evac_times):.2f}s")

## 7. Parameter sweep — compare agent counts

In [ ]:
import matplotlib.pyplot as plt
scenario_fresh = load_scenario("scenarios/corridor_simple.json")
counts = [5, 10, 20, 30]
results_by_count = {}

for n in counts:
    scenario_fresh.set_agent_count("jps-distributions_0", n)
    r = run_scenario(scenario_fresh)
    results_by_count[n] = r.evacuation_time
    r.cleanup()
    print(f"  {n} agents: {r.evacuation_time:.2f}s")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([str(n) for n in counts], [results_by_count[n] for n in counts])
ax.set_xlabel("Number of agents")
ax.set_ylabel("Evacuation time (s)")
ax.set_title("Evacuation time vs agent count")
plt.tight_layout()
plt.show()

## Cleanup

In [ ]:
result.cleanup()
print("Done.")